# 04 — Random Forest Yield Model
Trains a Random Forest Regressor on 2005–2020, validates on 2021–2024,
and predicts 2025 yield for all 5 states at 4 forecast dates.
Output: `outputs/predictions.csv`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

df = pd.read_csv("../data/processed/training_features.csv")
print(f"Loaded {len(df)} rows")
print(df.columns.tolist())

In [ ]:
# One-hot encode state
df = pd.get_dummies(df, columns=['state'])

# Define features (X) and target (Y)
TARGET = 'yield_bu_acre'
EXCLUDE = [TARGET, 'year']
FEATURES = [c for c in df.columns if c not in EXCLUDE]

# Train/test split by year
train = df[df['year'] <= 2020]
test  = df[(df['year'] >= 2021) & (df['year'] <= 2024)]

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f"Train: {len(train)} rows | Test: {len(test)} rows")

In [ ]:
# Train model
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42
)
model.fit(X_train, y_train)

# Validate
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f"Validation RMSE: {rmse:.2f} bu/acre")

In [ ]:
# Predict 2025
# Note: 2025 row must exist in training_features.csv with weather features
# but NO yield value (that's what we're predicting)

pred_2025 = df[df['year'] == 2025].copy()

if len(pred_2025) == 0:
    print("No 2025 rows found — add 2025 weather features to training_features.csv first.")
else:
    X_2025 = pred_2025[FEATURES]
    pred_2025['predicted_yield'] = model.predict(X_2025)

    # Bootstrap uncertainty (500 iterations)
    boot_preds = []
    for _ in range(500):
        idx = np.random.choice(len(X_train), len(X_train), replace=True)
        m = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=None)
        m.fit(X_train.iloc[idx], y_train.iloc[idx])
        boot_preds.append(m.predict(X_2025))

    boot_preds = np.array(boot_preds)
    pred_2025['ci_lower'] = np.percentile(boot_preds, 5, axis=0)
    pred_2025['ci_upper'] = np.percentile(boot_preds, 95, axis=0)

    pred_2025.to_csv("../outputs/predictions.csv", index=False)
    print(pred_2025[['year', 'predicted_yield', 'ci_lower', 'ci_upper']])